# 02 — Educational Exploratory Data Analysis

This notebook explores the filtered ASSISTments 2009–2010 Skill Builder dataset created in notebook 01. The analysis focuses on educational and product questions that inform leakage-safe longitudinal feature engineering, model evaluation, and teacher-facing reporting.

The one-hot `Skill_*` columns are the authoritative skill representation. `skill_name` is retained only as optional descriptive metadata and is not intended for modeling.


## Analysis questions

1. How often are original problems answered correctly?
2. How much interaction history is available per student?
3. Which skills are most practiced and which appear most difficult?
4. Does correctness change with additional opportunities?
5. How heterogeneous is student performance?
6. What do current-interaction help-seeking fields look like descriptively?
7. How severe are student, student-skill, and class cold-start limitations?

This is descriptive analysis. Associations should not be interpreted causally.


## Imports and notebook setup


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:,.3f}".format)

PLOT_TEMPLATE = "plotly_white"
COLOR_CORRECT = "#2A9D8F"
COLOR_INCORRECT = "#E76F51"

print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")
print(f"NumPy:  {np.__version__}")


Python: 3.13.14
pandas: 3.0.5
NumPy:  2.5.2


## Project paths and data loading

The path helper allows the notebook to run from either the repository root or the `notebooks/` directory.


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the nearest parent directory containing pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml.")


PROJECT_ROOT = find_project_root()
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "skill_builder_data_filtered_onehot.csv"
)

assert DATA_PATH.is_file(), f"Processed dataset not found: {DATA_PATH}"
print(f"Project root: {PROJECT_ROOT}")
print(f"EDA dataset:  {DATA_PATH.relative_to(PROJECT_ROOT)}")


Project root: W:\Workstation ExtDrive\007 Data Science\003 Data Science Projects\2026_p019 assistments_2009_2010
EDA dataset:  data\processed\skill_builder_data_filtered_onehot.csv


In [3]:
df = pd.read_csv(DATA_PATH, low_memory=False)
skill_columns = sorted(
    (column for column in df.columns if column.startswith("Skill_")),
    key=lambda column: int(column.removeprefix("Skill_")),
)

required_columns = {
    "user_id",
    "order_id",
    "correct",
    "opportunity",
    "student_class_id",
}
missing_required = required_columns.difference(df.columns)
assert not missing_required, f"Missing required columns: {sorted(missing_required)}"
assert skill_columns, "No Skill_* indicator columns were found."
assert set(df["correct"].dropna().unique()).issubset({0, 1})
assert (df[skill_columns].sum(axis=1) >= 1).all()

skill_matrix = df[skill_columns].astype("uint8")

print(f"Rows:                 {len(df):,}")
print(f"Students:             {df['user_id'].nunique():,}")
print(f"Skill indicators:     {len(skill_columns):,}")
print(f"Named skill labels:   {df['skill_name'].nunique(dropna=True):,}")
print(f"Missing skill labels: {df['skill_name'].isna().sum():,}")
print(f"Date/order range:      {df['order_id'].min():,} to {df['order_id'].max():,}")


Rows:                 259,386
Students:             4,163
Skill indicators:     123
Named skill labels:   101
Missing skill labels: 7,994
Date/order range:      20,224,180 to 38,310,202


## Dataset snapshot

This compact table establishes the population used throughout the notebook without repeating the full data-quality work from notebook 01.


In [4]:
overview = pd.Series(
    {
        "Interactions": len(df),
        "Students": df["user_id"].nunique(),
        "Classes": df["student_class_id"].nunique(),
        "Problems": df["problem_id"].nunique(),
        "Skill indicators": len(skill_columns),
        "Named skill labels": df["skill_name"].nunique(dropna=True),
        "Overall correctness": df["correct"].mean(),
    },
    name="Value",
).to_frame()
display(overview)

display(df.head())


,Value
Interactions,"259,386.000"
Students,"4,163.000"
Classes,247.000
Problems,"15,920.000"
Skill indicators,123.000
Named skill labels,101.000
Overall correctness,0.658


,user_id,order_id,skill_name,row_id,assignment_id,assistment_id,problem_id,original,attempt_count,ms_first_response,tutor_mode,answer_type,sequence_id,student_class_id,position,base_sequence_id,teacher_id,school_id,hint_count,hint_total,overlap_time,template_id,first_action,bottom_hint,opportunity,...,Skill_311,Skill_312,Skill_314,Skill_317,Skill_321,Skill_322,Skill_323,Skill_324,Skill_325,Skill_331,Skill_333,Skill_334,Skill_340,Skill_343,Skill_346,Skill_348,Skill_350,Skill_356,Skill_362,Skill_365,Skill_367,Skill_368,Skill_371,Skill_375,Skill_378
0,14,21617623,Circle Graph,3958,263599,53412,93383,1,1,26271,tutor,algebra,7118,12495,1,7118,42972,1,2,2,41131,52570,1,1,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,14,21617632,Circle Graph,3959,263599,53436,93407,1,1,29123,tutor,algebra,7118,12495,1,7118,42972,1,0,2,29123,52570,0,0,2,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,14,21617641,Circle Graph,3960,263599,53429,93400,1,1,13779,tutor,algebra,7118,12495,1,7118,42972,1,2,2,19905,52570,1,1,3,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,14,21617650,Circle Graph,3961,263599,53448,93419,1,1,16901,tutor,algebra,7118,12495,1,7118,42972,1,2,2,22600,52570,1,1,4,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,14,21617659,Circle Graph,3962,263599,53449,93420,1,1,11079,tutor,algebra,7118,12495,1,7118,42972,1,2,2,19704,52570,1,1,5,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 1. Overall first-attempt success

The target balance establishes the majority-class baseline and provides context for later ROC-AUC, PR-AUC, log-loss, Brier score, and calibration results.


In [5]:
outcome_summary = (
    df["correct"]
    .value_counts()
    .rename_axis("correct")
    .reset_index(name="interactions")
)
outcome_summary["outcome"] = outcome_summary["correct"].map(
    {0: "Incorrect", 1: "Correct"}
)
outcome_summary["share"] = outcome_summary["interactions"] / len(df)
display(outcome_summary[["outcome", "interactions", "share"]])

fig = px.bar(
    outcome_summary,
    x="outcome",
    y="interactions",
    color="outcome",
    color_discrete_map={
        "Correct": COLOR_CORRECT,
        "Incorrect": COLOR_INCORRECT,
    },
    text_auto=",",
    title="Overall correctness on original problems",
    labels={"outcome": "", "interactions": "Interactions"},
    template=PLOT_TEMPLATE,
)
fig.update_layout(showlegend=False)
fig.show()


,outcome,interactions,share
0,Correct,170678,0.658
1,Incorrect,88708,0.342


## 2. Student activity and performance heterogeneity

Students with little prior activity will require population-level fallback features. Student-level accuracy is shown only for students meeting a minimum-history threshold so that unstable one- or two-row rates do not dominate the chart.


In [6]:
student_summary = df.groupby("user_id").agg(
    interactions=("correct", "size"),
    correct=("correct", "sum"),
    accuracy=("correct", "mean"),
    classes=("student_class_id", "nunique"),
)

student_skill_presence = (
    skill_matrix
    .assign(user_id=df["user_id"].to_numpy())
    .groupby("user_id")[skill_columns]
    .max()
)
student_summary["skills_practiced"] = student_skill_presence.sum(axis=1)

student_quantiles = student_summary[
    ["interactions", "skills_practiced", "accuracy"]
].quantile([0, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00])
display(student_quantiles)


C:\Users\helsh\AppData\Local\Temp\ipykernel_3020\1846780012.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .assign(user_id=df["user_id"].to_numpy())


,interactions,skills_practiced,accuracy
0.000,1.000,1.000,0.000
0.100,3.000,1.000,0.223
0.250,8.000,2.000,0.487
0.500,19.000,4.000,0.667
0.750,55.000,10.000,0.806
0.900,150.000,25.000,0.941
0.950,314.000,45.000,1.000
0.990,599.040,75.380,1.000
1.000,963.000,92.000,1.000


In [7]:
fig = px.histogram(
    student_summary.reset_index(),
    x="interactions",
    nbins=60,
    log_y=True,
    title="Distribution of interactions per student",
    labels={"interactions": "Interactions per student", "count": "Students"},
    template=PLOT_TEMPLATE,
)
fig.add_vline(
    x=student_summary["interactions"].median(),
    line_dash="dash",
    annotation_text="Median",
)
fig.show()

history_threshold = 20
students_with_history = student_summary.query(
    "interactions >= @history_threshold"
).copy()

fig = px.histogram(
    students_with_history.reset_index(),
    x="accuracy",
    nbins=30,
    title=f"Student correctness with at least {history_threshold} interactions",
    labels={"accuracy": "Student-level correctness", "count": "Students"},
    template=PLOT_TEMPLATE,
)
fig.update_xaxes(tickformat=".0%")
fig.show()

print(
    f"{len(students_with_history):,} of {len(student_summary):,} students "
    f"have at least {history_threshold} interactions."
)


2,062 of 4,163 students have at least 20 interactions.


## 3. Skill activity and apparent difficulty

A row may contain more than one skill indicator. Each indicated skill receives credit for that interaction in this descriptive summary. To avoid unstable rankings, difficulty charts require minimum interaction and student counts.


In [8]:
skill_rows = skill_matrix.sum(axis=0)
skill_correct = skill_matrix.mul(df["correct"], axis=0).sum(axis=0)
skill_students = student_skill_presence.sum(axis=0)

skill_summary = pd.DataFrame(
    {
        "skill": skill_columns,
        "interactions": skill_rows.reindex(skill_columns).to_numpy(),
        "students": skill_students.reindex(skill_columns).to_numpy(),
        "correct": skill_correct.reindex(skill_columns).to_numpy(),
    }
)
skill_summary["incorrect"] = (
    skill_summary["interactions"] - skill_summary["correct"]
)
skill_summary["correctness"] = (
    skill_summary["correct"] / skill_summary["interactions"]
)

print("Skill-volume distribution")
display(
    skill_summary[["interactions", "students", "correctness"]]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
)


Skill-volume distribution


,interactions,students,correctness
count,123.000,123.000,123.000
mean,"2,538.358",341.309,0.617
std,"3,375.513",316.461,0.185
min,1.000,1.000,0.000
10%,78.000,14.200,0.375
25%,257.500,87.500,0.537
50%,"1,144.000",264.000,0.648
75%,"3,680.000",475.000,0.751
90%,"6,866.000",789.400,0.820
max,"18,739.000","1,353.000",0.950


In [11]:
top_skill_volume = skill_summary.nlargest(20, "interactions").sort_values(
    "interactions"
)
fig = px.bar(
    top_skill_volume,
    x="interactions",
    y="skill",
    orientation="h",
    title="Twenty most frequently represented skills",
    labels={"interactions": "Interactions", "skill": "Skill ID"},
    template=PLOT_TEMPLATE,
)
fig.update_yaxes(
    tickmode="array",
    tickvals=top_skill_volume["skill"],
    ticktext=top_skill_volume["skill"],
)
fig.update_layout(height=700, margin={"l": 100})
fig.show()


In [12]:
MIN_SKILL_INTERACTIONS = 200
MIN_SKILL_STUDENTS = 50
eligible_skills = skill_summary.query(
    "interactions >= @MIN_SKILL_INTERACTIONS and students >= @MIN_SKILL_STUDENTS"
).copy()

lowest_skills = eligible_skills.nsmallest(15, "correctness").sort_values(
    "correctness", ascending=False
)
highest_skills = eligible_skills.nlargest(15, "correctness").sort_values(
    "correctness"
)

fig = px.bar(
    lowest_skills,
    x="correctness",
    y="skill",
    orientation="h",
    hover_data=["interactions", "students"],
    title=(
        "Lowest observed correctness among sufficiently represented skills "
        f"(≥{MIN_SKILL_INTERACTIONS} interactions, ≥{MIN_SKILL_STUDENTS} students)"
    ),
    labels={"correctness": "Observed correctness", "skill": "Skill ID"},
    template=PLOT_TEMPLATE,
)
fig.update_xaxes(tickformat=".0%")
fig.show()

print(f"{len(eligible_skills)} of {len(skill_summary)} skills meet the display thresholds.")
display(
    eligible_skills.nsmallest(15, "correctness")[
        ["skill", "interactions", "students", "correctness"]
    ]
)


91 of 123 skills meet the display thresholds.


,skill,interactions,students,correctness
84,Skill_292,427,163,0.136
91,Skill_299,491,135,0.316
70,Skill_166,389,88,0.368
83,Skill_290,459,176,0.373
47,Skill_76,3050,483,0.381
106,Skill_325,1983,332,0.410
24,Skill_37,3455,408,0.458
51,Skill_81,3315,626,0.461
13,Skill_17,7453,452,0.499
85,Skill_293,464,194,0.500


### Optional readable labels

`skill_name` can support reports and EDA, but it is not assumed to be a one-to-one mapping to the one-hot skill IDs. The table below describes the available labels separately and does not use them as model features.


In [13]:
skill_name_summary = (
    df.dropna(subset=["skill_name"])
    .groupby("skill_name")
    .agg(
        interactions=("correct", "size"),
        students=("user_id", "nunique"),
        correctness=("correct", "mean"),
    )
    .sort_values("interactions", ascending=False)
)
display(skill_name_summary.head(20))


,interactions,students,correctness
skill_name,,,
Conversion of Fraction Decimals Percents,18739,1225,0.637
Equation Solving Two or Fewer Steps,17311,961,0.646
Addition and Subtraction Integers,12741,1226,0.599
Addition and Subtraction Fractions,11332,1353,0.677
Proportion,8063,671,0.647
Ordering Positive Decimals,7317,942,0.750
Multiplication and Division Integers,6874,900,0.788
Table,6834,713,0.739
Pythagorean Theorem,6611,283,0.628


## 4. Practice opportunity and learning curves

These curves describe aggregate correctness by the current opportunity number. They do not isolate causal learning effects: student selection, problem difficulty, skill mix, and dropout may all change across opportunities.


In [14]:
MAX_OPPORTUNITY_SHOWN = 30
MIN_CURVE_ROWS = 100

overall_curve = (
    df.groupby("opportunity")
    .agg(
        correctness=("correct", "mean"),
        interactions=("correct", "size"),
        students=("user_id", "nunique"),
    )
    .reset_index()
    .query(
        "opportunity <= @MAX_OPPORTUNITY_SHOWN and interactions >= @MIN_CURVE_ROWS"
    )
)

fig = px.line(
    overall_curve,
    x="opportunity",
    y="correctness",
    markers=True,
    hover_data=["interactions", "students"],
    title="Aggregate correctness by opportunity number",
    labels={"opportunity": "Opportunity", "correctness": "Observed correctness"},
    template=PLOT_TEMPLATE,
)
fig.update_yaxes(tickformat=".0%")
fig.show()


In [10]:
TOP_SKILLS_FOR_CURVES = 5
MAX_SKILL_OPPORTUNITY = 20
MIN_SKILL_CURVE_ROWS = 30
curve_frames = []

for skill in skill_summary.nlargest(TOP_SKILLS_FOR_CURVES, "interactions")["skill"]:
    curve = (
        df.loc[skill_matrix[skill].eq(1)]
        .groupby("opportunity")
        .agg(
            correctness=("correct", "mean"),
            interactions=("correct", "size"),
            students=("user_id", "nunique"),
        )
        .reset_index()
        .query(
            "opportunity <= @MAX_SKILL_OPPORTUNITY "
            "and interactions >= @MIN_SKILL_CURVE_ROWS"
        )
    )
    curve["skill"] = skill
    curve_frames.append(curve)

skill_curves = pd.concat(curve_frames, ignore_index=True)

fig = px.line(
    skill_curves,
    x="opportunity",
    y="correctness",
    color="skill",
    markers=True,
    hover_data=["interactions", "students"],
    title=f"Learning curves for the {TOP_SKILLS_FOR_CURVES} highest-volume skills",
    labels={"opportunity": "Opportunity", "correctness": "Observed correctness"},
    template=PLOT_TEMPLATE,
)
fig.update_yaxes(tickformat=".0%")
fig.show()


## 5. Class coverage

Class-level reporting needs minimum-size rules. Very small class identifiers should not be used for stable comparisons or public-facing rankings.


In [15]:
class_summary = df.groupby("student_class_id").agg(
    students=("user_id", "nunique"),
    interactions=("correct", "size"),
    skills=("skill_name", "nunique"),
    correctness=("correct", "mean"),
)

display(
    class_summary[["students", "interactions", "correctness"]]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
)

fig = px.histogram(
    class_summary.reset_index(),
    x="students",
    nbins=40,
    title="Distribution of students per class ID",
    labels={"students": "Unique students", "count": "Classes"},
    template=PLOT_TEMPLATE,
)
fig.show()


,students,interactions,correctness
count,247.000,247.000,247.000
mean,19.178,"1,050.146",0.593
std,21.913,"2,819.637",0.245
min,1.000,1.000,0.000
10%,1.000,2.000,0.245
25%,1.000,15.500,0.490
50%,12.000,104.000,0.628
75%,28.000,844.500,0.750
90%,52.400,"2,242.000",0.876
95%,60.000,"4,081.800",1.000


## 6. Help-seeking and response behavior

The following fields describe the current interaction and therefore **must not be used directly to predict that same row's `correct` target**. They are explored descriptively here and may later be transformed into shifted historical features. Differences are associations, not causal effects.


In [16]:
hint_bins = pd.cut(
    df["hint_count"].fillna(0),
    bins=[-0.1, 0.5, 1.5, 2.5, np.inf],
    labels=["0", "1", "2", "3+"],
)
hint_summary = (
    df.assign(hint_group=hint_bins)
    .groupby("hint_group", observed=True)
    .agg(
        interactions=("correct", "size"),
        correctness=("correct", "mean"),
    )
    .reset_index()
)
display(hint_summary)

fig = px.bar(
    hint_summary,
    x="hint_group",
    y="correctness",
    text_auto=".1%",
    hover_data=["interactions"],
    title="Current-interaction correctness by hint count",
    labels={"hint_group": "Current hint count", "correctness": "Observed correctness"},
    template=PLOT_TEMPLATE,
)
fig.update_yaxes(tickformat=".0%")
fig.show()


C:\Users\helsh\AppData\Local\Temp\ipykernel_3020\661739087.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.assign(hint_group=hint_bins)


,hint_group,interactions,correctness
0,0,222153,0.768
1,1,4785,0.000
2,2,7454,0.000
3,3+,24994,0.000


In [17]:
behavior_columns = [
    "attempt_count",
    "hint_count",
    "bottom_hint",
    "ms_first_response",
]
available_behavior_columns = [
    column for column in behavior_columns if column in df.columns
]
behavior_summary = df[available_behavior_columns].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
).T
display(behavior_summary)

if "first_action" in df.columns:
    first_action_summary = (
        df.assign(first_action_label=df["first_action"].astype("string").fillna("Missing"))
        .groupby("first_action_label")
        .agg(interactions=("correct", "size"), correctness=("correct", "mean"))
        .query("interactions >= 100")
        .sort_values("interactions", ascending=False)
    )
    display(first_action_summary)


,count,mean,std,min,50%,75%,90%,95%,99%,max
attempt_count,"259,386.000",1.579,13.433,0.000,1.000,1.000,2.000,3.000,8.000,"3,824.000"
hint_count,"259,386.000",0.437,1.171,0.000,0.000,0.000,2.000,4.000,5.000,7.000
bottom_hint,"259,386.000",0.096,0.294,0.000,0.000,0.000,0.000,1.000,1.000,1.000
ms_first_response,"259,386.000","50,337.005","373,596.283",0.000,"21,665.000","48,219.000","98,957.500","151,347.250","366,366.000","84,076,920.000"


C:\Users\helsh\AppData\Local\Temp\ipykernel_3020\2199179499.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.assign(first_action_label=df["first_action"].astype("string").fillna("Missing"))


,interactions,correctness
first_action_label,,
0,235030,0.726
1,18133,0.000
2,6223,0.000


## 7. Cold-start and student-skill sparsity

Student-skill counts include every indicated skill on multi-skill rows. These distributions inform smoothing, population priors, minimum-history rules, and evaluation by history length.


In [18]:
student_skill_counts = (
    skill_matrix
    .assign(user_id=df["user_id"].to_numpy())
    .groupby("user_id")[skill_columns]
    .sum()
    .stack()
    .rename("interactions")
)
student_skill_counts = student_skill_counts[student_skill_counts.gt(0)]

student_history_thresholds = pd.DataFrame(
    {
        "minimum_interactions": [1, 2, 5, 10, 20, 50, 100],
    }
)
student_history_thresholds["students"] = student_history_thresholds[
    "minimum_interactions"
].map(lambda threshold: student_summary["interactions"].ge(threshold).sum())
student_history_thresholds["student_share"] = (
    student_history_thresholds["students"] / len(student_summary)
)

display(student_history_thresholds)
display(
    student_skill_counts.describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame()
)


C:\Users\helsh\AppData\Local\Temp\ipykernel_3020\1922948333.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .assign(user_id=df["user_id"].to_numpy())


,minimum_interactions,students,student_share
0,1,4163,1.000
1,2,4022,0.966
2,5,3628,0.871
3,10,2968,0.713
4,20,2062,0.495
5,50,1097,0.264
6,100,668,0.160


,interactions
count,"41,981.000"
mean,7.437
std,9.104
min,1.000
10%,1.000
25%,2.000
50%,5.000
75%,10.000
90%,17.000
95%,23.000


In [19]:
pair_plot_data = student_skill_counts.clip(upper=50).reset_index()
pair_plot_data.columns = ["user_id", "skill", "interactions"]

fig = px.histogram(
    pair_plot_data,
    x="interactions",
    nbins=50,
    log_y=True,
    title="Student-skill interaction counts (values above 50 clipped for display)",
    labels={"interactions": "Interactions per student-skill pair", "count": "Pairs"},
    template=PLOT_TEMPLATE,
)
fig.show()


## 8. EDA summary for downstream design

The table below records the principal sample-size facts that should guide feature engineering and evaluation. It intentionally avoids making claims about causality or psychological mastery.


In [20]:
eda_summary = pd.Series(
    {
        "Interactions": len(df),
        "Students": len(student_summary),
        "Overall correctness": df["correct"].mean(),
        "Median interactions per student": student_summary["interactions"].median(),
        "Students with >=20 interactions": student_summary["interactions"].ge(20).sum(),
        "Skill indicators": len(skill_summary),
        "Skills meeting EDA display thresholds": len(eligible_skills),
        "Student-skill pairs": len(student_skill_counts),
        "Median interactions per student-skill pair": student_skill_counts.median(),
        "Classes": len(class_summary),
        "Median students per class": class_summary["students"].median(),
    },
    name="Value",
).to_frame()
display(eda_summary)


,Value
Interactions,"259,386.000"
Students,"4,163.000"
Overall correctness,0.658
Median interactions per student,19.000
Students with >=20 interactions,"2,062.000"
Skill indicators,123.000
Skills meeting EDA display thresholds,91.000
Student-skill pairs,"41,981.000"
Median interactions per student-skill pair,5.000
Classes,247.000


## Recommended decisions to carry forward

- Retain early interactions using population and skill-level fallback features rather than dropping all cold-start rows.
- Smooth historical rates when student or student-skill history is short.
- Compute every historical behavioral feature with a shift so interaction `t` uses only information from interactions before `t`.
- Evaluate performance by student-history length and skill support, not only overall.
- Use student-clustered bootstrap intervals because rows from the same student are dependent.
- Apply minimum sample-size rules to skill and class reports.
- Treat support thresholds as demonstration thresholds, not validated educational cutoffs.

Next: define and test the leakage-safe longitudinal feature table in notebook 03.
